## Install Required Libraries

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
!pip install -q datasets transformers evaluate sentence-transformers openai pandas numpy tqdm python-dotenv  google-generativeai requests tqdm -q sacrebleu rouge_score bert_score anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 248.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 29.3 MB/s eta 0:00:00


In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.3/390.3 kB 30.0 MB/s eta 0:00:00


## Import Required Libraries

In [ ]:
import os
import csv
import json
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Other Packages
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm.auto import tqdm
import os
import csv

## Display Settings

In [1]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', None)           # Don't wrap to next line
pd.set_option('display.max_colwidth', None)    # Show full column content

## Hugginface Login

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in using environment variable")
else:
    print("No HF_TOKEN found")

Logged in using environment variable


## OpenAI Login

In [ ]:
from openai import OpenAI
import os

# OpenAI API key
api_key = userdata.get('OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

## Mount Google Drive

In [ ]:
from google.colab import drive

# mount drive
drive.mount('/content/drive')

## Load the test split and choose sample size

In [ ]:
dataset_name = "Lakshan2003/customer-support-client-agent-conversations"
split_name   = "test"
num_samples  = None  # Set to an integer to evaluate a subset (in order) or None for all

test_dataset = load_dataset(dataset_name, split=split_name)
if num_samples is not None:
    test_dataset = test_dataset.select(range(num_samples))  # Keep original order

print(f"Loaded {len(test_dataset)} test samples")

### Select and Load a LoRA‑finetuned model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

lora_adapter = "Lakshan2003/Phi-4-mini-instruct-customerservice"

# Load the base SmolLM3 model
base_model_name = "microsoft/Phi-4-mini-instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=False
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load LoRA adapter on top
model = PeftModel.from_pretrained(
    base_model,
    lora_adapter
)

# Merge the adapter
model = model.merge_and_unload()

# Set to eval mode
model.eval()

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/35.7M [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_feat

### Prompt Templat for the Model

In [ ]:
ft_prompt = """<|system|>{instruction}<|end|>
<|user|>Summarized Conversation History:
{history}

Client Question:
{client_question}<|end|>
<|assistant|>"""

# End of sentence special token
EOS_TOKEN = tokenizer.eos_token

### Generation Settings

In [ ]:
# Sampling parameters (top_p, top_k, etc.)
generation_config = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "do_sample": True,
    "top_p": 0.9,
    "top_k": 50,
}

### Build Chat template function

In [ ]:
def build_prompt(example):
    # accept dict rows or raw strings
    if isinstance(example, str):
        example = {"client_question": example}
    elif not isinstance(example, dict):
        example = {"client_question": str(example)}

    return ft_prompt.format(
        instruction=example.get("instruction", ""),
        history=example.get("history_summary", ""),
        client_question=example.get("client_question", "")
    )

### Output file

In [ ]:
lora_adapter = "Phi-4-mini-instruct-customerservice"
results_dir  = f"/content/drive/MyDrive/fyp-2025/Datasets/TestData/{lora_adapter}"
results_path = os.path.join(results_dir, f"results_lora_customerservice{lora_adapter}.csv")
os.makedirs(results_dir, exist_ok=True)

In [ ]:
model.eval()
torch.set_grad_enabled(False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

In [ ]:
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token

eos_ids = [tokenizer.eos_token_id]
end = "<|end|>"
if hasattr(tokenizer, "get_vocab") and end in tokenizer.get_vocab():
    end_id = tokenizer.convert_tokens_to_ids(end)
    if end_id != tokenizer.eos_token_id:
        eos_ids = [tokenizer.eos_token_id, end_id]

In [ ]:
SAVE_COLS = [
    "conversation_id",
    "instruction",
    "history_summary",
    "client_question",
    "ground_truth",        # from refined_agent_answer
    "generated_answer",
]

In [ ]:
def as_str(x):
    try:
        return str(x)
    except Exception:
        return ""

In [ ]:
# resume Logic
processed_ids, header_written = set(), False
if os.path.exists(results_path):
    try:
        prev = pd.read_csv(results_path)
        if "conversation_id" in prev.columns:
            processed_ids = set(prev["conversation_id"].astype(str))
        header_written = True
        print(f"Resuming from {len(processed_ids)} saved rows.")
    except Exception as e:
        print(f"CSV unreadable, starting fresh: {e}")

In [ ]:
to_run = []
for ex in test_dataset:  # expects dict-like rows
    cid = as_str(ex.get("conversation_id", ""))
    if cid and cid in processed_ids:
        continue
    to_run.append(ex)
print(f"Pending: {len(to_run)}")

Pending: 36669


In [ ]:
gen_batch_size = 64
pbar = tqdm(total=len(to_run), desc="Generating", unit="rec")

model.eval()
torch.set_grad_enabled(False)

for start in range(0, len(to_run), gen_batch_size):
    batch = to_run[start : start + gen_batch_size]

    prompts, metas = [], []
    for ex in batch:
        instruction     = ex.get("instruction", "")
        history_turns   = ex.get("conversation_history", "")
        history         = ex.get("history_summary", "")
        client_question = ex.get("client_question", "")
        ground_truth    = ex.get("refined_agent_answer", "")

        input_prompt = ft_prompt.format(
            instruction=instruction,
            history=history,
            client_question=client_question
        )
        prompts.append(input_prompt)
        metas.append({
            "conversation_id": str(ex.get("conversation_id", "")),
            "instruction": instruction,
            "history_summary": history,
            "history": history_turns,
            "client_question": client_question,
            "ground_truth": ground_truth,
        })

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            **generation_config,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
            return_dict_in_generate=True,
            use_cache=True,
        )

    seqs = out.sequences
    start_idx = enc.input_ids.shape[1]  # robust slice point for the whole batch
    gen_texts = [
        tokenizer.decode(seqs[i, start_idx:], skip_special_tokens=True).strip()
        for i in range(seqs.size(0))
    ]

    out_df = pd.DataFrame(
        [{**m, "generated_answer": g} for m, g in zip(metas, gen_texts)],
        columns=SAVE_COLS
    )
    out_df.to_csv(
        results_path,
        mode="a",
        header=not os.path.exists(results_path),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pbar.update(len(batch))

pbar.close()

Generating:   0%|          | 0/36669 [00:00<?, ?rec/s]

In [ ]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv(results_path)

# Convert to Dataset
dataset = Dataset.from_pandas(df)

# Define repo name with model + use case
repo_id = "Lakshan2003/Phi-4-mini-customerservice-evaldata"

# Push to Hub
dataset.push_to_hub(repo_id, private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  24%|##3       | 5.11MB / 21.7MB            

README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-evaldata/commit/1b43deed3cc9ffecf4acf923f1871e576e85ff4f', commit_message='Upload dataset', commit_description='', oid='1b43deed3cc9ffecf4acf923f1871e576e85ff4f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-evaldata', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Phi-4-mini-customerservice-evaldata'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd

# Dataset names
src_repo = "Lakshan2003/GPT-4.1-customerservice-evaldata"   # has history
dst_repo = "Lakshan2003/Phi-4-mini-customerservice-evaldata" # needs history

# Load datasets
src = load_dataset(src_repo, split="train")
dst = load_dataset(dst_repo, split="train")

# Convert to DataFrame
src_df = src.to_pandas()[["conversation_id", "history"]]
dst_df = dst.to_pandas()

# Merge by conversation_id
merged = dst_df.merge(src_df, on="conversation_id", how="left")

# Insert 'history' after 'instruction'
cols = list(merged.columns)
cols.remove("history")
idx = cols.index("instruction") + 1
cols.insert(idx, "history")
merged = merged[cols]

# Convert back to Dataset and push
final = Dataset.from_pandas(merged)
final.push_to_hub(dst_repo, commit_message="Added 'history' column")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  80%|#######9  | 33.7MB / 42.2MB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-evaldata/commit/3f4d0c18d70e4297260ca95e529f3cd14502fb08', commit_message="Added 'history' column", commit_description='', oid='3f4d0c18d70e4297260ca95e529f3cd14502fb08', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-evaldata', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Phi-4-mini-customerservice-evaldata'), pr_revision=None, pr_num=None)

# Lexical Overlap Scores Calculation

In [ ]:
df = pd.read_csv(results_path)
preds = df["generated_answer"].tolist()
refs  = df["ground_truth"].tolist()

## Bleu Score Calculation

In [ ]:
# Lexical overlap metrics
bleu   = evaluate.load("bleu").compute(predictions=preds, references=[[r] for r in refs])["bleu"]

bleu

0.21582900227017687

## Google Bleu Score Calculation

In [ ]:
gbleu  = evaluate.load("google_bleu").compute(predictions=preds, references=refs)["google_bleu"]

gbleu

0.22662229694082686

## Meteor Score Calculation

In [ ]:
meteor = evaluate.load("meteor").compute(predictions=preds, references=refs)["meteor"]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
print(meteor)

0.43029954230628537


## Rougue Score Calculation

In [ ]:
rouge  = evaluate.load("rouge").compute(predictions=preds, references=refs)

rouge

{'rouge1': np.float64(0.460740646068739),
 'rouge2': np.float64(0.24203904478943372),
 'rougeL': np.float64(0.3746840686698035),
 'rougeLsum': np.float64(0.3747026202356415)}

### Summarized Lexical Overlap Results

In [ ]:
import pandas as pd

# Build a summary table
metrics_table = pd.DataFrame([
    {"metric": "BLEU",         "score": bleu},
    {"metric": "Google BLEU",  "score": gbleu},
    {"metric": "METEOR",       "score": meteor},
    {"metric": "ROUGE-1",      "score": rouge["rouge1"]},
    {"metric": "ROUGE-2",      "score": rouge["rouge2"]},
    {"metric": "ROUGE-L",      "score": rouge["rougeL"]},
])

In [ ]:
metrics_table

,metric,score
0,BLEU,0.215829
1,Google BLEU,0.226622
2,METEOR,0.430300
3,ROUGE-1,0.460741
4,ROUGE-2,0.242039
5,ROUGE-L,0.374684


In [ ]:
from datasets import Dataset

repo = "Lakshan2003/Lexical-Overlap-Results-Phi-4-mini-customerservice"

hf_dataset = Dataset.from_pandas(metrics_table.reset_index(drop=True))

# Push to the hub
hf_dataset.push_to_hub(repo)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.18kB / 1.18kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-Phi-4-mini-customerservice/commit/5ee99dc02a2428624a04905493c0b26ea1e79992', commit_message='Upload dataset', commit_description='', oid='5ee99dc02a2428624a04905493c0b26ea1e79992', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-Phi-4-mini-customerservice', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Lexical-Overlap-Results-Phi-4-mini-customerservice'), pr_revision=None, pr_num=None)

## Semantic Similarity Metrics

In [ ]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=results_path, split="train")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'history_summary', 'client_question', 'ground_truth', 'generated_answer'],
    num_rows: 36669
})

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Generate embeddings for Groundtruth and generated answer

In [ ]:
def embed_batch(batch):
    gen = [str(x) if x is not None else "" for x in batch["generated_answer"]]
    gt  = [str(x) if x is not None else "" for x in batch["ground_truth"]]

    # normalize_embeddings=True => unit vectors, so cosine = dot product
    gen_emb = model.encode(gen, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    gt_emb  = model.encode(gt,  convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    # store as lists
    return {
        "generated_answer_embedding_transformers": [e.tolist() for e in gen_emb],
        "ground_truth_embedding_transformers": [e.tolist() for e in gt_emb],
    }

ds = ds.map(embed_batch, batched=True, batch_size=256)
ds

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'history_summary', 'client_question', 'ground_truth', 'generated_answer', 'generated_answer_embedding_transformers', 'ground_truth_embedding_transformers'],
    num_rows: 36669
})

In [ ]:
import numpy as np

def cosine_batch(batch):
    gen = np.array(batch["generated_answer_embedding_transformers"], dtype=np.float32)
    gt  = np.array(batch["ground_truth_embedding_transformers"], dtype=np.float32)

    # cosine = sum(gen * gt) (using normalized embeddings)
    cos = (gen * gt).sum(axis=1)
    return {"cosine_similarity": cos.tolist()}

ds = ds.map(cosine_batch, batched=True, batch_size=1024)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

### BERT Score

In [ ]:
import evaluate
bertscore = evaluate.load("bertscore")

def add_bertscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_answer"]]
    refs  = [str(x) if x else "" for x in batch["ground_truth"]]
    result = bertscore.compute(predictions=preds, references=refs, lang="en")
    return {
        "bertscore_precision": result["precision"],
        "bertscore_recall": result["recall"],
        "bertscore_f1": result["f1"],
    }

ds = ds.map(add_bertscore, batched=True, batch_size=64)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### BART Score

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)
bart_model.eval()

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def compute_bartscore(sources, targets):
    scores = []
    for src, tgt in zip(sources, targets):
        inputs = bart_tokenizer(src, return_tensors="pt", truncation=True, max_length=512).to(device)
        with bart_tokenizer.as_target_tokenizer():
            labels = bart_tokenizer(tgt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            loss = bart_model(**inputs, labels=labels["input_ids"]).loss
        scores.append(-loss.item())  # negative loss = BARTScore
    return scores

def add_bartscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_answer"]]
    refs  = [str(x) if x else "" for x in batch["ground_truth"]]
    ref_hyp = compute_bartscore(refs, preds)
    hyp_ref = compute_bartscore(preds, refs)
    mean_score = [(a + b) / 2 for a, b in zip(ref_hyp, hyp_ref)]
    return {
        "bartscore_ref_hyp": ref_hyp,
        "bartscore_hyp_ref": hyp_ref,
        "bartscore_mean": mean_score,
    }

In [ ]:
ds = ds.map(add_bartscore, batched=True, batch_size=128)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
df = ds.to_pandas()

In [ ]:
print("Summary:")
print(f"Mean cosine similarity : {df['cosine_similarity'].mean():.4f}")
print(f"Mean BERTScore F1      : {df['bertscore_f1'].mean():.4f}")
print(f"Mean BARTScore (mean)  : {df['bartscore_mean'].mean():.4f}")

Summary:
Mean cosine similarity : 0.6891
Mean BERTScore F1      : 0.9107
Mean BARTScore (mean)  : -2.2872


In [ ]:
import os
import pandas as pd

MODEL_NAME = "Phi-4-mini"
SAVE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results"

metrics_rows = [
    {"model": MODEL_NAME, "metric": "BLEU",        "value": float(bleu)},
    {"model": MODEL_NAME, "metric": "Google BLEU", "value": float(gbleu)},
    {"model": MODEL_NAME, "metric": "METEOR",      "value": float(meteor)},
    {"model": MODEL_NAME, "metric": "ROUGE-1",     "value": float(rouge["rouge1"])},
    {"model": MODEL_NAME, "metric": "ROUGE-2",     "value": float(rouge["rouge2"])},
    {"model": MODEL_NAME, "metric": "ROUGE-L",     "value": float(rouge["rougeL"])},
    {"model": MODEL_NAME, "metric": "Cosine Similarity (mean)", "value": float(df["cosine_similarity"].mean())},
    {"model": MODEL_NAME, "metric": "BERTScore F1 (mean)",      "value": float(df["bertscore_f1"].mean())},
    {"model": MODEL_NAME, "metric": "BARTScore (mean)",         "value": float(df["bartscore_mean"].mean())},
]

metrics_table = pd.DataFrame(metrics_rows)

os.makedirs(SAVE_DIR, exist_ok=True)
csv_path = f"{SAVE_DIR}/metrics_{MODEL_NAME}.csv"
metrics_table.to_csv(csv_path, index=False)

print(csv_path)

/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results/metrics_Phi-4-mini.csv


## Context-Aware similarities

### Conditional BERT Score

In [ ]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=results_path, split="train")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'history_summary', 'client_question', 'ground_truth', 'generated_answer'],
    num_rows: 36669
})

In [ ]:
import evaluate

conditional_bertscore_metric = evaluate.load("bertscore")

def add_conditional_bertscore(batch):
    conditioned_predictions = []
    conditioned_references  = []

    for hist, question, gen_ans, gt_ans in zip(
        batch["history_summary"],
        batch["client_question"],
        batch["generated_answer"],
        batch["ground_truth"]
    ):
        hist = str(hist) if hist else ""
        question = str(question) if question else ""
        gen_ans = str(gen_ans) if gen_ans else ""
        gt_ans  = str(gt_ans) if gt_ans else ""

        condition = (
            "Conversation History: " + hist.strip() + "\n"
            "Client Question: " + question.strip()
        )

        conditioned_predictions.append(
            condition + "\nAnswer: " + gen_ans
        )

        conditioned_references.append(
            condition + "\nAnswer: " + gt_ans
        )

    scores = conditional_bertscore_metric.compute(
        predictions=conditioned_predictions,
        references=conditioned_references,
        lang="en"
    )

    return {
        "conditional_bertscore_precision": scores["precision"],
        "conditional_bertscore_recall": scores["recall"],
        "conditional_bertscore_f1": scores["f1"],
    }

ds = ds.map(add_conditional_bertscore, batched=True, batch_size=64)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Context BART Score

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn"
).to(device)

bart_model.eval()

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def compute_bartscore(source_texts, target_texts):
    scores = []

    for src, tgt in zip(source_texts, target_texts):
        inputs = bart_tokenizer(
            src,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with bart_tokenizer.as_target_tokenizer():
            labels = bart_tokenizer(
                tgt,
                return_tensors="pt",
                truncation=True,
                max_length=1024
            ).to(device)

        with torch.no_grad():
            loss = bart_model(
                **inputs,
                labels=labels["input_ids"]
            ).loss

        # Negative log-likelihood = BARTScore
        scores.append(-loss.item())

    return scores

In [ ]:
def add_context_aware_bartscore(batch):
    source_texts = []
    generated_targets = []
    reference_targets = []

    for hist, question, gen_ans, gt_ans in zip(
        batch["history_summary"],
        batch["client_question"],
        batch["generated_answer"],
        batch["ground_truth"],
    ):
        hist = str(hist) if hist else ""
        question = str(question) if question else ""
        gen_ans = str(gen_ans) if gen_ans else ""
        gt_ans  = str(gt_ans) if gt_ans else ""

        context_prefix = (
            "Conversation History: " + hist.strip() + "\n"
            "Client Question: " + question.strip()
        )

        # Same source for both directions
        source_texts.append(context_prefix)

        generated_targets.append(gen_ans)
        reference_targets.append(gt_ans)

    # Reference → Generated
    bartscore_ref_to_gen = compute_bartscore(
        source_texts,
        generated_targets
    )

    # Reference → Ground truth
    bartscore_ref_to_gt = compute_bartscore(
        source_texts,
        reference_targets
    )

    bartscore_mean = [
        (a + b) / 2
        for a, b in zip(bartscore_ref_to_gen, bartscore_ref_to_gt)
    ]

    return {
        "bartscore_ctx_ref_gen": bartscore_ref_to_gen,
        "bartscore_ctx_ref_gt": bartscore_ref_to_gt,
        "bartscore_ctx_mean": bartscore_mean,
    }

In [ ]:
ds = ds.map(add_context_aware_bartscore, batched=True, batch_size=128)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


#### Context Aware Cosine Similarity

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

sentence_encoder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def embed_context_aware_sentences(batch):
    generated_texts = []
    reference_texts = []

    for hist, question, gen_ans, gt_ans in zip(
        batch["history_summary"],
        batch["client_question"],
        batch["generated_answer"],
        batch["ground_truth"],
    ):
        hist = str(hist) if hist else ""
        question = str(question) if question else ""
        gen_ans = str(gen_ans) if gen_ans else ""
        gt_ans  = str(gt_ans) if gt_ans else ""

        context_prefix = (
            "Conversation History: " + hist.strip() + "\n"
            "Client Question: " + question.strip()
        )

        generated_texts.append(
            context_prefix + "\nAnswer: " + gen_ans
        )

        reference_texts.append(
            context_prefix + "\nAnswer: " + gt_ans
        )

    generated_embeddings = sentence_encoder.encode(
        generated_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=False
    )

    reference_embeddings = sentence_encoder.encode(
        reference_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=False
    )

    return {
        "context_aware_generated_embedding": [e.tolist() for e in generated_embeddings],
        "context_aware_reference_embedding": [e.tolist() for e in reference_embeddings],
    }

ds = ds.map(embed_context_aware_sentences, batched=True, batch_size=256)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

In [ ]:
import numpy as np

def context_aware_cosine_similarity(batch):
    gen_emb = np.array(
        batch["context_aware_generated_embedding"],
        dtype=np.float32
    )
    ref_emb = np.array(
        batch["context_aware_reference_embedding"],
        dtype=np.float32
    )

    cosine_scores = (gen_emb * ref_emb).sum(axis=1)

    return {
        "context_aware_cosine_similarity": cosine_scores.tolist()
    }

ds = ds.map(context_aware_cosine_similarity, batched=True, batch_size=1024)

Map:   0%|          | 0/36669 [00:00<?, ? examples/s]

In [ ]:
df = ds.to_pandas()

In [ ]:
print("Context-aware Evaluation Summary:")

print(f"Mean cosine similarity (context-aware sentence embeddings) : "
      f"{df['context_aware_cosine_similarity'].mean():.4f}")

print(f"Mean BERTScore F1 (context-aware)                          : "
      f"{df['conditional_bertscore_f1'].mean():.4f}")

print(f"Mean BARTScore (context-aware mean)                        : "
      f"{df['bartscore_ctx_mean'].mean():.4f}")

Context-aware Evaluation Summary:
Mean cosine similarity (context-aware sentence embeddings) : 0.9707
Mean BERTScore F1 (context-aware)                          : 0.9774
Mean BARTScore (context-aware mean)                        : -2.7430


In [ ]:
import os
import pandas as pd

MODEL_NAME = "Phi-4-mini"
SAVE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results"

context_metrics_rows = [
    {
        "model": MODEL_NAME,
        "metric": "Cosine Similarity (context-aware sentence embeddings)",
        "value": float(df["context_aware_cosine_similarity"].mean())
    },
    {
        "model": MODEL_NAME,
        "metric": "BERTScore F1 (context-aware)",
        "value": float(df["conditional_bertscore_f1"].mean())
    },
    {
        "model": MODEL_NAME,
        "metric": "BARTScore (context-aware mean)",
        "value": float(df["bartscore_ctx_mean"].mean())
    },
]

context_metrics_table = pd.DataFrame(context_metrics_rows)

os.makedirs(SAVE_DIR, exist_ok=True)
csv_path = f"{SAVE_DIR}/metrics_context_aware_{MODEL_NAME}.csv"
context_metrics_table.to_csv(csv_path, index=False)

print(csv_path)

/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results/metrics_context_aware_Phi-4-mini.csv


### LLM as a judge Evaluation

In [ ]:
import os
import time
import json
import pandas as pd
from anthropic import Anthropic

#### Claude API Setup

In [ ]:
from google.colab import userdata
claude_api = userdata.get('claude_api')

In [ ]:
anthropic = Anthropic(api_key=claude_api)

In [ ]:
# Configuration
original_csv = "/content/drive/MyDrive/fyp-2025/Datasets/LLMJudgeData/Phi-4-mini-customerservice-evaldata.csv"

processed_folder = "/content/drive/MyDrive/fyp-2025/Datasets/LLMJudgeProcessedData/"
processed_csv = processed_folder + "Phi-4-mini-customerservice-evaldata.csv"

batch_size = 50
batch_pause = 1.5  # seconds

#### EVALUATION PROMPT TEMPLATE (outside loop)

In [ ]:
EVAL_PROMPT = """
You are an expert evaluator specializing in customer-service interactions.
Evaluate the Generated Response using the Client Question and Conversation History summary as context, along with a Reference Agent Response provided only as a high-quality example.
The Reference Agent Response is provided only as guidance to illustrate what a professional, contextually appropriate answer might look like.
The Generated Response should NOT replicate or closely mirror it.
Instead, it should demonstrate human-like fluency, contextual understanding, and professionalism while maintaining natural variation in expression and tone.
Your task is to assess how human-like, contextually aware, and professionally appropriate the Generated Response is.

Note:
The Conversation History Summary represents the main context that was used when generating the response.
The full Conversation History is provided only as additional background information to help you better understand the situation if needed.

Context Inputs:
Conversation History: {history}
Conversation Summarized History: {history_summary}
Client Question: {client_question}
Reference Agent Response (for guidance only): {ground_truth}
Generated Response: {generated_answer}

Evaluation Criteria and Scoring (1–5 each):

1. Human-Likeness:
This checks how natural and fluent the Generated Response sounds in normal spoken conversation.
It looks at flow, rhythm, and how close it feels to real human speech.

Rating Scale:
1 = Highly robotic or unnatural
2 = Noticeably rigid or scripted
3 = Generally natural but somewhat stiff
4 = Natural and conversational
5 = Fully natural, smooth, and human-like

2. Continuity and Context Understanding:
This evaluates how well the Generated Response integrates with the preceding conversation whether it maintains continuity,
references earlier details accurately, and demonstrates awareness of situational context.

Rating Scale:
1 = Ignores or contradicts context
2 = Uses context incorrectly or inconsistently
3 = Uses context but with noticeable gaps
4 = Accurate and consistent use of context
5 = Fully coherent, precise integration of context

3. Tone and Clarity:
This measures verbal tone, emotional intelligence, and clarity of expression.
It assesses professionalism, empathy, politeness, and phrasing appropriateness for a spoken customer-service exchange.

Rating Scale:
1 = Unprofessional or unclear
2 = Understandable but flat or loosely structured
3 = Clear and appropriate, with standard professionalism
4 = Professional, well-structured, and concise
5 = Highly polished, clear, respectful, and well-balanced

4. Task Appropriateness:
This evaluates whether the Generated Response successfully and completely addresses the client’s stated need,
while maintaining procedural accuracy typical of a service agent.

Rating Scale:
1 = Does not address the client’s request
2 = Addresses request incompletely
3 = Provides an adequate response
4 = Fully addresses the request
5 = Fully addresses the request and adds meaningful helpful value

Return your answer as valid JSON only.
Do not include any explanation, commentary, additional text, or markdown formatting.
Output must contain JSON only and nothing else.All the below keys and there judgement score should be included in your json response. Strictly follow only below json output.always provide the score for all tasks in the json.

Output Format (return only JSON):
{{
  "Human-Likeness": [1–5],
  "Continuity-and-Context-Understanding": [1–5],
  "Tone-and-Clarity": [1–5],
  "Task-Appropriateness": [1–5]
}}
"""

In [ ]:
# Load Original or Resume From Processed Copy
os.makedirs(processed_folder, exist_ok=True)

if os.path.exists(processed_csv):
    print("Existing processed file found. Resuming from previous progress...")
    df = pd.read_csv(processed_csv)
else:
    print("No processed copy found. Creating one...")
    df = pd.read_csv(original_csv)

    # Add scoring columns if missing
    score_cols = [
        "Human-Likeness",
        "Continuity and Context Understanding",
        "Tone and Clarity",
        "Task Appropriateness"
    ]
    for c in score_cols:
        if c not in df.columns:
            df[c] = ""

    df.to_csv(processed_csv, index=False, encoding="utf-8-sig")

print("Loaded rows:", len(df))

Existing processed file found. Resuming from previous progress...
Loaded rows: 6000


In [ ]:
# 5. Identify Rows That Need Processing
mask = df["Human-Likeness"].isna() | (df["Human-Likeness"] == "")
remaining_rows = df[mask]

start_index = len(df) - len(remaining_rows)
print(f"Starting evaluation from row {start_index}/{len(df)}\n")

batch_counter = 0

Starting evaluation from row 6000/6000



In [ ]:
def safe_get(d, keys, row_idx, field_name):
    """
    d: the JSON dict returned by Claude
    keys: list of alternative keys to try
    """
    for k in keys:
        if k in d:
            return d[k]
    raise KeyError(f"Missing '{field_name}' in JSON at row {row_idx}. JSON keys returned: {list(d.keys())}")

#### MAIN LOOP

In [ ]:
from tqdm import tqdm
import re
import json

# 6. MAIN EVALUATION LOOP WITH TQDM
for idx in tqdm(remaining_rows.index, desc="Evaluating rows", unit="row"):

    row = df.loc[idx]

    prompt_filled = EVAL_PROMPT.format(
        history=row["history"],
        history_summary=row["history_summary"],
        client_question=row["client_question"],
        ground_truth=row["ground_truth"],
        generated_answer=row["generated_answer"],
    )

    try:
        # SEND REQUEST TO CLAUDE
        response = anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=500,
            temperature=0,
            messages=[{"role": "user", "content": prompt_filled}],
        )

        # EXTRACT RAW TEXT
        raw_text = response.content[0].text.strip()
        if not raw_text:
            raise ValueError("Claude returned an empty response")

        # CLEAN THE JSON TEXT
        cleaned = raw_text.strip()

        cleaned = re.sub(r"^```[a-zA-Z0-9]*\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
        cleaned = re.sub(r"<[^>]+>", "", cleaned).strip()
        cleaned = cleaned.replace("\ufeff", "").replace("\u200b", "").strip("`").strip()

        # TRY PARSING JSON
        try:
            result_json = json.loads(cleaned)
        except json.JSONDecodeError:
            raise ValueError(f"Invalid JSON from Claude:\n{raw_text}")

        # UPDATE DF ONLY IF SUCCESSFUL
        df.at[idx, "Human-Likeness"] = result_json["Human-Likeness"]
        df.at[idx, "Continuity and Context Understanding"] = result_json[
            "Continuity-and-Context-Understanding"
        ]
        df.at[idx, "Tone and Clarity"] = result_json["Tone-and-Clarity"]
        df.at[idx, "Task Appropriateness"] = result_json["Task-Appropriateness"]

        # SAVE BATCH (SAFE)
        batch_counter += 1

        if batch_counter >= batch_size:
            df.to_csv(processed_csv, index=False, encoding="utf-8-sig")
            print(f"Batch saved (processed up to row {idx}).")
            batch_counter = 0

    except Exception as e:
        # ON ANY ERROR, STOP IMMEDIATELY
        # DO NOT SAVE ANYTHING
        print(f"\nERROR at row {idx}: {e}")
        print("Stopping execution WITHOUT saving this partial batch.")
        break

    time.sleep(batch_pause)

Evaluating rows:   2%|▏         | 49/3000 [03:28<3:22:09,  4.11s/row]

Batch saved (processed up to row 3049).


Evaluating rows:   3%|▎         | 99/3000 [07:02<3:24:20,  4.23s/row]

Batch saved (processed up to row 3099).


Evaluating rows:   5%|▍         | 149/3000 [10:37<3:49:11,  4.82s/row]

...

Evaluating rows:  98%|█████████▊| 2949/3000 [3:44:08<03:19,  3.91s/row]

Batch saved (processed up to row 5949).


Evaluating rows: 100%|█████████▉| 2999/3000 [3:47:45<00:04,  4.17s/row]

Batch saved (processed up to row 5999).


Evaluating rows: 100%|██████████| 3000/3000 [3:47:50<00:00,  4.56s/row]


In [ ]:
df = pd.read_csv(processed_csv)

In [ ]:
from datasets import Dataset

repo = "Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data"

# Convert to HF Dataset (remove index column if exists)
hf_dataset = Dataset.from_pandas(df.reset_index(drop=True))

# Push to the hub (creates parquet automatically)
hf_dataset.push_to_hub(repo, private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          | 35.9kB / 6.93MB            

README.md:   0%|          | 0.00/786 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data/commit/8bdfa6228211514b3bb244b23323317795355cd7', commit_message='Upload dataset', commit_description='', oid='8bdfa6228211514b3bb244b23323317795355cd7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load dataset from Hugging Face
dataset = load_dataset(
    "Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data",
    split="train"
)

# Convert to pandas DataFrame
df = dataset.to_pandas()

# Correct column names (exact match)
task_columns = [
    "Human-Likeness",
    "Continuity and Context Understanding",
    "Tone and Clarity",
    "Task Appropriateness",
]

# Ensure numeric dtype
df[task_columns] = df[task_columns].apply(pd.to_numeric, errors="coerce")

# Compute task-wise mean
task_wise_mean = df[task_columns].mean()

# Convert to clean table
task_wise_mean_df = task_wise_mean.reset_index()
task_wise_mean_df.columns = ["Task", "Mean Score"]

README.md:   0%|          | 0.00/790 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.93M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [ ]:
print(task_wise_mean_df)

                                   Task  Mean Score
0                        Human-Likeness    3.987667
1  Continuity and Context Understanding    3.359167
2                      Tone and Clarity    4.034000
3                  Task Appropriateness    3.093000
